In [10]:
import datetime

from app.data.dto.main.BunkeringStep import BunkeringStep
from app.services.db_service import DbService
from app.services.external_api.searoute_api import SearouteApi

from dotenv import load_dotenv

load_dotenv()

sql_db_service = DbService()
await sql_db_service.init_pool()
searoute_api = SearouteApi("https://api.searoutes.com/", "EWhTo2x2hihNDCPZjCaMgFDWGegJoVLnYP7mqi5L")
route, err = await sql_db_service.get_route_by_id("f5a102f0-5769-4e85-a947-fdb0199a67e8")
departure_port, err = await sql_db_service.get_port_by_locode("RULED")
destination_port, err = await sql_db_service.get_port_by_locode("AEJEA")


In [12]:
from app.services.utils import utils
import math
import asyncio
import datetime
from typing import List, Dict, Optional, Tuple
from concurrent.futures import ProcessPoolExecutor

# -------------------- utils --------------------

def adjust_from_weekend(date: datetime.date) -> datetime.date:
    while date.weekday() >= 5:
        date -= datetime.timedelta(days=1)
    return date


# -------------------- fuel prices --------------------

async def find_fuel_price(
    sql_db_service,
    port,
    fuel_name: str,
    date: datetime.date,
) -> Optional[float]:
    date = adjust_from_weekend(date)

    price_db, _ = await sql_db_service.get_port_fuel_price_by_port_locode(
        port.locode, fuel_name, date
    )
    if price_db:
        if price_db.value > 0:
            return price_db.value

    alt_ids, err = await sql_db_service.get_alternative_mabux_ids(port.locode.strip())
    if err or not alt_ids:
        return None

    for mabux_id in alt_ids:
        price_db, _ = await sql_db_service.get_port_fuel_price_by_port_mabux_id(
            mabux_id, fuel_name, date
        )
        if price_db:
            if price_db.value > 0:
                return price_db.value

    return None


# -------------------- port pricing --------------------

async def build_priced_port(
    sql_db_service,
    port,
    fuels: list,
    price_date: datetime.date,
    semaphore: asyncio.Semaphore,
) -> Optional[Dict]:
    async with semaphore:
        tasks = {
            fuel.name: find_fuel_price(
                sql_db_service,
                port,
                fuel.name,
                price_date,
            )
            for fuel in fuels
        }

        prices = await asyncio.gather(*tasks.values())

        fuel_info = {}
        prices_count = 0
        prices_sum = 0.0

        for fuel_name, price in zip(tasks.keys(), prices):
            if price is not None:
                prices_count += 1
                prices_sum += price

            fuel_info[fuel_name] = {
                "fuel_name": fuel_name,
                "fuel_price": price,
                "available": price is not None,
                "quantity": None,
            }

        if prices_count == 0:
            return None

        return {
            "port": port,
            "fuel_info": fuel_info,
            "prices_count": prices_count,
            "prices_sum": prices_sum,
            "_distance_km": getattr(port, "_distance_km", None),
            "marked": False,
        }


# -------------------- coordinates chunking --------------------

def chunk_coords(coords, step: int, chunk_size: int):
    sampled = coords[::step]
    for i in range(0, len(sampled), chunk_size):
        yield sampled[i : i + chunk_size]


# -------------------- ports search (parallel over chunks) --------------------

def find_nearest_waypoint(step, waypoints):
    nearest_wp = None
    min_dist = float("inf")

    for wp in waypoints:
        d = haversine_km(
            step.get("port").latitude,
            step.get("port").longitude,
            wp.latitude,
            wp.longitude,
        )
        if d < min_dist:
            min_dist = d
            nearest_wp = wp

    return nearest_wp

def haversine_km(lat1, lon1, lat2, lon2) -> float:
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(math.radians(lat1))
        * math.cos(math.radians(lat2))
        * math.sin(dlon / 2) ** 2
    )
    return 2 * R * math.asin(math.sqrt(a))

async def collect_ports_parallel(
    sea_route,
    sql_db_service,
    radius_km: float = 500.0,
    limit: int = 50,
    step: int = 50,
    chunk_size: int = 20,
    max_parallel_chunks: int = 4,
):
    seen = {}

    semaphore = asyncio.Semaphore(max_parallel_chunks)

    async def process_chunk(chunk):
        async with semaphore:
            for c in chunk:
                ports, err = await sql_db_service.search_ports_nearby_with_prices(
                    c.latitude,
                    c.longitude,
                    n=limit,
                    radius_km=radius_km,
                )
                if err or not ports:
                    continue

                for p in ports:
                    if p.locode not in seen:
                        seen[p.locode] = p

    tasks = [
        process_chunk(chunk)
        for chunk in chunk_coords(sea_route.seaRouteCoordinates, step, chunk_size)
    ]

    await asyncio.gather(*tasks)

    ports = list(seen.values())
    ports.sort(key=lambda p: getattr(p, "_distance_km", float("inf")))
    utils.unique_ports(ports)
    return ports

def find_nearest_coord_index(step, coordinates) -> int:
    min_dist = float("inf")
    min_idx = 0

    for i, c in enumerate(coordinates):
        d = haversine_km(
            step.get("port").latitude,
            step.get("port").longitude,
            c.latitude,
            c.longitude,
        )
        if d < min_dist:
            min_dist = d
            min_idx = i

    return min_idx


def distance_from_start_to_index(coordinates, idx: int) -> float:
    total = 0.0

    for i in range(1, idx + 1):
        prev = coordinates[i - 1]
        curr = coordinates[i]
        total += haversine_km(
            prev.latitude,
            prev.longitude,
            curr.latitude,
            curr.longitude,
        )

    return total

def enrich_ports_with_eta_and_distance(
    steps: List[dict],
    sea_route,
    top_n_marked: int = 3
) -> None:
    coordinates = sea_route.seaRouteCoordinates
    waypoints = sea_route.waypoints

    for step in steps:
        # --- ETA from nearest waypoint ---
        wp = find_nearest_waypoint(step, waypoints)
        step["eta_datetime"] = wp.eta_datetime if wp else None

        # --- distance along route ---
        idx = find_nearest_coord_index(step, coordinates)
        step["distance"] = distance_from_start_to_index(coordinates, idx)

    steps.sort(key=lambda s: s["prices_sum"])
    for s in steps[:top_n_marked]:
        s["marked"] = True

    steps.sort(key=lambda s: s["distance"])

    # normalize numbering
    for i, s in enumerate(steps, start=1):
        s["n"] = i

# -------------------- bunkering steps --------------------

async def build_bunkering_steps(
    sql_db_service,
    ports: List,
    fuels: list,
    price_date: datetime.date,
    max_concurrency: int = 40,
    top_n_marked: int = 3,
    steps_limit: int = 20,
) -> List[Dict]:
    semaphore = asyncio.Semaphore(max_concurrency)

    tasks = [
        build_priced_port(
            sql_db_service,
            port=p,
            fuels=fuels,
            price_date=price_date,
            semaphore=semaphore,
        )
        for p in ports
    ]

    results = await asyncio.gather(*tasks)
    steps = [r for r in results if r is not None]

    # sort: (a) prices_count desc, (b) prices_sum asc
    steps.sort(key=lambda s: (-s["prices_count"], s["prices_sum"]))

    return steps[:steps_limit]


# -------------------- full pipeline --------------------

async def build_bunkering_plan_fast(
    searoute_api,
    sql_db_service,
    departure_port,
    destination_port,
    fuels,
    speed_kts: float = 10.0,
):
    sea_route, err = await searoute_api.build_sea_route(
        departure_port.latitude,
        departure_port.longitude,
        destination_port.latitude,
        destination_port.longitude,
        speed_in_knots=speed_kts,
        is_plan=True,
        departure_dt=datetime.datetime.now(),
    )
      await self.sql_db_service.create_event(Event.error(
                user_id=session.user_id,
                payload={
                    "error": err,
                },
            ))
        raise RuntimeError(err)

    ports = await collect_ports_parallel(
        sea_route=sea_route,
        sql_db_service=sql_db_service,
    )

    ports = [departure_port, *ports, destination_port]

    steps_dicts = await build_bunkering_steps(
        sql_db_service=sql_db_service,
        ports=ports,
        fuels=fuels,
        price_date=datetime.date.today(),
    )

    enrich_ports_with_eta_and_distance(steps_dicts, sea_route)

     # Convert dicts to BunkeringStep objects
    bunkering_steps: List[BunkeringStep] = []
    for idx, step in enumerate(steps_dicts, start=1):
        bunkering_step = BunkeringStep(
            n=idx,
            port=step["port"],
            eta_datetime=step.get("eta_datetime"),
            distance=step.get("distance"),
            fuel_info=step["fuel_info"],
            agent_required=False,       # can adjust later if needed
            selected=False,
            to_show=True,
            marked=step.get("marked", False)
        )
        bunkering_steps.append(bunkering_step)


    return bunkering_steps


In [13]:
bunkering_steps = await build_bunkering_plan_fast(
    searoute_api=searoute_api,
    sql_db_service=sql_db_service,
    departure_port=departure_port,
    destination_port=destination_port,
    fuels=route.fuels,
)
len(bunkering_steps)

20

[BunkeringStep(n=1, port=SeaPortDB(locode='NLVSN', country_code='', country_name='Netherlands', port_name='Velsen-Noord', latitude=52.469295, longitude=4.631525, rank_score=None, similarity_score=None, combined_score=None, match_type='unknown', mabux_ids=[104, 535], port_size=None, mabux_id=None, barge_status=None, truck_status=None, agent_contact_list=None, bubble_id='1713117935850x901088188619565800', search_key='velsen-noord|nlvsn|netherlands|', id='ddf5c0c2-96c4-4d77-8ad3-7f0412e6f312'), eta_datetime=datetime.datetime(2026, 1, 13, 18, 3, 20, 740000), distance=2021.693415457581, fuel_info={'VLS FO': {'fuel_name': 'VLS FO', 'fuel_price': 407.0, 'available': True, 'quantity': None}, 'MGO LS': {'fuel_name': 'MGO LS', 'fuel_price': 612.0, 'available': True, 'quantity': None}}, agent_required=False, selected=False, to_show=True, marked=False),
 BunkeringStep(n=2, port=SeaPortDB(locode='NLVEL', country_code='', country_name='Netherlands', port_name='Velsen', latitude=52.46574, longitude=4

In [5]:
price = await find_fuel_price(sql_db_service, s['port'], "VLS FO", datetime.datetime.now().date())
price

NameError: name 's' is not defined